In [2]:
# cella per importare le librerie necessarie

import pandas as pd
import geopandas as gpd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import datasets

from  sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import f1_score

In [9]:
from pathlib  import Path
data_path = Path('../data/raw')

files = {'grid':'trentino-grid.geojson',
         'adm_reg':'administrative_regions_Trentino.json',
        'weather':'meteotrentino-weather-station-data.json',
        'precip':'precipitation-trentino.csv',
        'precip-avail':'precipitation-trentino-data-availability.csv',
        'SET-1':'SET-nov-2013.csv',
        'SET-2':'SET-dec-2013.csv',
        'SET-lines':'line.csv',
        'twitter':'social-pulse-trentino.geojson'}

In [23]:
# Creo il DataFrame geopandas con i dati meteo

import json
from shapely.geometry import Point
df_grid = gpd.read_file(data_path / files['grid'])

with open(data_path / files['weather']) as f:
    weather_json = json.load(f)

weather_df = gpd.GeoDataFrame(weather_json['features'])

# La colonna del vento è divisa nella forma forza_del_vento@direzione_azimutale. Vogliamo separare questi due dati in due diverse colonne
winds_cols = [c for c in weather_df.columns if c.startswith("winds.")]

new_cols = []

for c in winds_cols:

    split_cols = weather_df[c].str.split("@", n=1, expand=True)

    # ensure two columns always exist
    split_cols = split_cols.reindex(columns=[0, 1])

    split_cols.columns = [f"{c}_spd", f"{c}_dir"]

    split_cols = split_cols.apply(pd.to_numeric, errors="coerce")

    new_cols.append(split_cols)

weather_df = pd.concat([weather_df] + new_cols, axis=1)
weather_df.drop(columns=winds_cols,inplace=True)

# Introduco la colonna geometry, che contiene le posizioni delle stazioni meteo di rilevamento
weather_df['geometry'] = weather_df['geomPoint.geom'].apply(lambda x:Point(x['coordinates'][0], x['coordinates'][1]))
weather_df.drop(columns=['geomPoint.geom'],inplace=True)

weather_df

,station,elevation,date,timestamp,minTemperature,maxTemperature,precipitation,minWind,maxWind,temperatures.0000,...,winds.2245_dir,winds.2300_spd,winds.2300_dir,winds.2315_spd,winds.2315_dir,winds.2330_spd,winds.2330_dir,winds.2345_spd,winds.2345_dir,geometry
0,T0071,905,2013-11-01,1383260400,4.5,12.3,False,0.0,2.5,8.4,...,248.0,NaN,NaN,0.0,201.0,0.0,199.0,NaN,NaN,POINT (10.79582897 46.31340453)
1,T0032,1155,2013-11-01,1383260400,6.5,10.2,False,NaN,NaN,7.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (11.25371981 45.94027203)
2,T0096,1205,2013-11-01,1383260400,6.5,11.8,False,NaN,NaN,8.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (11.6645808 46.38363633)
3,T0074,720,2013-11-01,1383260400,6.2,13.6,False,0.0,4.8,10.2,...,345.0,1.0,244.0,0.5,192.0,0.0,157.0,NaN,NaN,POINT (10.91841055 46.35159801)
4,T0101,201,2013-11-01,1383260400,11.1,16.3,False,NaN,NaN,12.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (11.07973339 46.15635256)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2190,T0431,1055,2013-12-31,1388444400,-5.5,0.5,False,NaN,NaN,-3.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (11.33626355 46.26304408)
2191,T0428,525,2013-12-31,1388444400,-2.2,6.9,False,NaN,NaN,-0.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (10.64240944 45.94061606)
2192,T0437,1465,2013-12-31,1388444400,-9.9,-2.4,False,0.1,2.5,-8.3,...,115.0,1.4,110.0,1.4,98.0,1.5,109.0,1.5,106.0,POINT (11.76685208 46.47831772)
2193,T0469,801,2013-12-31,1388444400,-4.1,2.7,False,0.1,1.4,-2.5,...,349.0,0.7,340.0,0.4,353.0,0.8,353.0,0.7,15.0,POINT (11.6299111 46.05717778)


In [21]:
# modifica del dataframe in modo da mettere in ordine cronologico

meteo_df = None

for station_id in weather_df['station'].drop_duplicates():

    df_station = weather_df[weather_df["station"] == station_id]

    # === COLONNE PRECIPITAZIONI ===
    temp_cols = [c for c in df_station.columns if c.startswith("temperatures.")]
    prec_cols = [c for c in df_station.columns if c.startswith("precipitations.")]
    winds_cols_spd = [c for c in df_station.columns if (c.startswith("winds.") and c.endswith("_spd"))]
    winds_cols_dir = [c for c in df_station.columns if (c.startswith("winds.") and c.endswith("_dir"))]

    # lista finale
    rows = []
    prec = []
    winds_spd = []
    winds_dir = []

    # === COSTRUZIONE NUOVO DATAFRAME ===
    for _, row in df_station.iterrows():

        date = pd.to_datetime(row["date"])

        for i in range(len(temp_cols)):

            # estrae HHMM
            hhmm = temp_cols[i].split(".")[1]

            hour = hhmm[:2]
            minute = hhmm[2:]

            # costruzione datetime
            dt = date.replace(
                hour=int(hour),
                minute=int(minute)
            )

            # formato richiesto:
            # minuto-ora-giorno-mese-anno
            # esempio 1215140313
            custom_time = dt.strftime("%M%H%d%m%y")

            rows.append({
                'datetime': custom_time,
                'temperatures_' + station_id: row[temp_cols[i]],
                'precipitations_' + station_id: row[prec_cols[i]],
                'winds_spd_' + station_id: row[winds_cols_spd[i]],
                'winds_dir_' + station_id: row[winds_cols_dir[i]]
            }) 
    
    df = pd.DataFrame(rows)

    if meteo_df is None:
        meteo_df = df
    else:
        meteo_df = meteo_df.merge(df, on="datetime")

meteo_df

,datetime,temperatures_T0071,precipitations_T0071,winds_spd_T0071,winds_dir_T0071,temperatures_T0032,precipitations_T0032,winds_spd_T0032,winds_dir_T0032,temperatures_T0096,...,winds_spd_T0437,winds_dir_T0437,temperatures_T0469,precipitations_T0469,winds_spd_T0469,winds_dir_T0469,temperatures_T0450,precipitations_T0450,winds_spd_T0450,winds_dir_T0450
0,0000011113,8.4,0.0,0.1,205.0,7.3,0.0,NaN,NaN,8.2,...,0.0,289.0,9.7,0.0,1.1,173.0,3.6,0.0,0.7,273.0
1,1500011113,8.3,0.0,NaN,NaN,7.3,0.0,NaN,NaN,8.2,...,0.3,115.0,9.4,0.0,1.0,169.0,3.9,0.0,0.6,258.0
2,3000011113,8.2,0.0,0.0,256.0,7.1,0.0,NaN,NaN,8.1,...,0.2,328.0,9.5,0.0,0.4,155.0,4.0,0.0,0.7,266.0
3,4500011113,8.1,0.0,NaN,NaN,6.9,0.0,NaN,NaN,8.1,...,0.2,106.0,9.6,0.0,NaN,NaN,3.9,0.0,0.9,253.0
4,0001011113,8.2,0.0,0.5,231.0,6.7,0.0,NaN,NaN,8.1,...,0.9,92.0,9.6,0.0,0.5,203.0,3.8,0.0,0.9,263.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5755,4522311213,-6.6,0.0,1.7,243.0,-3.0,0.0,NaN,NaN,-9.1,...,1.0,115.0,-3.3,0.0,0.5,349.0,-9.0,0.0,0.6,297.0
5756,0023311213,-7.4,0.0,0.9,273.0,-3.0,0.0,NaN,NaN,-8.7,...,1.4,110.0,-3.3,0.0,0.7,340.0,-9.1,0.0,0.9,286.0
5757,1523311213,-7.3,0.0,1.3,255.0,-3.0,0.0,NaN,NaN,-9.1,...,1.4,98.0,-3.4,0.0,0.4,353.0,-9.1,0.0,0.8,287.0
5758,3023311213,-7.9,0.0,1.2,268.0,-3.1,0.0,NaN,NaN,-9.2,...,1.5,109.0,-3.6,0.0,0.8,353.0,-9.3,0.0,0.9,273.0


In [ ]:
# proviamo a guardare i dati APPA

data_path = Path('../data/external')

files_appa = {'dati_inq':'APPA_inquinamento_aria_Nov_Dec_2013.csv',
            'pos':'posizioni_stazioni_appa.csv'}

appa_df = pd.read_csv(data_path / files_appa['dati_inq'], encoding="latin1")
print(appa_df)

appa_pos = pd.read_csv(data_path / files_appa['pos'])
appa_pos


              Stazione Inquinante        Data  Ora Valore Unità di misura
0      Parco S. Chiara       PM10  2013-11-01    1     23           µg/mc
1      Parco S. Chiara       PM10  2013-11-01    2     25           µg/mc
2      Parco S. Chiara       PM10  2013-11-01    3     23           µg/mc
3      Parco S. Chiara       PM10  2013-11-01    4     21           µg/mc
4      Parco S. Chiara       PM10  2013-11-01    5     20           µg/mc
...                ...        ...         ...  ...    ...             ...
23120  Borgo Valsugana      Ozono  2013-12-31   20      6           µg/mc
23121  Borgo Valsugana      Ozono  2013-12-31   21      5           µg/mc
23122  Borgo Valsugana      Ozono  2013-12-31   22      4           µg/mc
23123  Borgo Valsugana      Ozono  2013-12-31   23      4           µg/mc
23124  Borgo Valsugana      Ozono  2013-12-31   24      4           µg/mc

[23125 rows x 6 columns]


,Nome stazione,Indirizzo,Dati stazione,Località,Zona,Tipologia,EU - codice europeo,IT - codice italiano,Posizione
0,PIANA ROTALIANA,Sette pergole,https://bollettino.appa.tn.it/aria/stazione/22,Mezzolombardo,Rurale,Fondo,IT1930A,402212,"46.19683, 11.11343"
1,RIVA GAR,via Trento,https://bollettino.appa.tn.it/aria/stazione/9,Riva del Garda,Suburbana,Fondo,IT0753A,402204,"45.89146, 10.84448"
2,MONTE GAZA,Malga Gaza,https://bollettino.appa.tn.it/aria/stazione/15,Vallelaghi,Rurale,Fondo,IT1191A,402203,"46.08253,10.95804"
3,TRENTO PSC,Parco S. Chiara,https://bollettino.appa.tn.it/aria/stazione/2,Trento,Urbana,Fondo,IT1037A,402209,"46.06292, 11.12620"
4,ROVERETO LGP,via Manzoni,https://bollettino.appa.tn.it/aria/stazione/6,Rovereto,Urbana,Fondo,IT0591A,402206,"45.89243, 11.03941"
5,TRENTO VBZ,via Bolzano,https://bollettino.appa.tn.it/aria/stazione/4,Trento,Urbana,Traffico,IT1859A,402211,"46.10433, 11.11022"
6,AVIO A22,loc. Ischiaforana,https://bollettino.appa.tn.it/aria/stazione/23,Avio,Rurale,Fondo,NaN,402213,"45.74215, 10.97043"
7,BORGO VAL,via 4 novembre,https://bollettino.appa.tn.it/aria/stazione/8,Borgo Valsugana,Suburbana,Fondo,IT0703A,402201,"46.05184,11.45389"


In [ ]:
def geom_appa(ring_series):
    target = np.array(['adult']*len(ring_series))
    target[ring_series <= 8] = 'young'
    target[ring_series >= 11] = 'old'
    